# Boosted Decision Tree

In [2]:
import pandas as pd
import numpy as np
import re
import math
import time
from datetime import timedelta
from pathlib import Path
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, make_scorer, roc_auc_score, accuracy_score,
)

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
"""datasets = {
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}"""

datasets = {
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
}


"""datasets = {
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv'
}"""


"""datasets = {
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
}"""

"datasets = {\n    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',\n    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'\n}"

# Training Duke


In [3]:
def training_duke(file_path: Path, csv_name: str):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    # --- Controllo colonne target attese ---
    required_targets = ["ER", "PR", "HER2"]
    missing = [c for c in required_targets if c not in df_validi.columns]
    if missing:
        print(f"[ERRORE] {csv_name} - mancano colonne target: {missing}")
        return None

    # --- Creo i target binari direttamente (già binari nel Duke) ---
    final_target_list = ["ER_class", "PR_class", "HER2_class"]

    df_validi["ER_class"]   = pd.to_numeric(df_validi["ER"], errors="coerce")
    df_validi["PR_class"]   = pd.to_numeric(df_validi["PR"], errors="coerce")
    df_validi["HER2_class"] = pd.to_numeric(df_validi["HER2"], errors="coerce")

    # Tengo solo righe con tutti i target presenti e casto a int
    df_validi = df_validi.dropna(subset=final_target_list).copy()
    for col in final_target_list:
        df_validi[col] = df_validi[col].astype(int)

    # --- Tolgo target con 1 sola classe ---
    targets_da_rimuovere = []
    for col in final_target_list:
        if df_validi[col].nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # --- Features / Target / Groups ---
    raw_target_cols = ["ER", "PR", "HER2"]

    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign", "GRADE", "isTN", "Breast"
    ] + raw_target_cols + final_target_list

    features = df_validi.drop(columns=features_to_drop, errors="ignore")
    target = df_validi[final_target_list]
    groups = df_validi["Patient ID"]

    # Imputazione features numeriche
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.fillna(features.mean(numeric_only=True))

    # Pulizia nomi colonne
    features.columns = [re.sub(r"\[|\]|<", "", col) for col in features.columns]

    # --- StratifiedGroupKFold ---
    if "HER2_class" in final_target_list:
        y_strat = target["HER2_class"].astype(str)
        n_pos = int(target["HER2_class"].sum())
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg("_".join, axis=1)
        n_splits = 5


    # Debug
    min_class_count = y_strat.value_counts().min()
    n_splits = min(n_splits, min_class_count)

    if n_splits < 2:
        print(f"[ERRORE] {csv_name} - troppo pochi campioni per CV stratificata")
        return None




    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    if len(rare) > 0:
        y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(sgkf.split(features, y_strat, groups=groups))

    # Debug utilissimo: controlla positivi HER2 per fold
    """if "HER2_class" in final_target_list:
        for k, (tr, te) in enumerate(splits):
            tr_pos = int(target.iloc[tr]["HER2_class"].sum())
            te_pos = int(target.iloc[te]["HER2_class"].sum())
            print(f"[{csv_name}] Fold {k}: HER2 train pos={tr_pos} | test pos={te_pos}")"""

    base_model = HistGradientBoostingClassifier(
        random_state=42,
        class_weight="balanced",
        early_stopping=False,
        l2_regularization=0.0,
        min_samples_leaf=10
    )
    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        "estimator__learning_rate": [0.05, 0.1],
        "estimator__max_iter": [100, 200],
        "estimator__max_depth": [3, 5],
    }

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average="macro", zero_division=0))
        return float(np.mean(scores))

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = math.prod(len(v) for v in iperparametri.values())
    #print(f"\nInizio Grid Search HistGradientBoosting (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=splits,
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=True,
        error_score="raise"
    )

    grid_search.fit(features, target)

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    clean_best_params = {k.replace("estimator__", ""): v for k, v in best_params.items()}

    final_params = {
        "random_state": 42,
        "class_weight": "balanced",
        "early_stopping": False,
        "l2_regularization": 0.0,
        "min_samples_leaf": 10,
        **clean_best_params
    }

    # Metriche per fold
    fold_reports = []
    for k,(train_idx, test_idx) in enumerate(splits):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = MultiOutputClassifier(HistGradientBoostingClassifier(**final_params))
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)



        # DEBUG: Fold k
        print(f"\n[DEBUG] Fold {k}")
        for i, col in enumerate(final_target_list):
            proba_i = y_proba_list[i]
            y_true_i = y_test.iloc[:, i].values

            print(
                f"  Target: {col} | "
                f"y_true classes: {np.unique(y_true_i)} | "
                f"proba shape: {proba_i.shape}"
            )

            

        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i].values
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            auc_val = np.nan
            if len(np.unique(y_true_i)) == 2:
                proba_i = y_proba_list[i]
                if proba_i.shape[1] == 2:
                    auc_val = roc_auc_score(y_true_i, proba_i[:, 1])

            fold_metrics[col] = {"f1": f1, "accuracy": acc, "auc": auc_val}


        # Debug
        print(f"\nMetriche Fold {k}")
        for col, m in fold_metrics.items():
            auc_str = "nan" if np.isnan(m["auc"]) else f"{m['auc']:.3f}"
            print(
                f"  {col}: "
                f"F1={m['f1']:.3f} | "
                f"ACC={m['accuracy']:.3f} | "
                f"AUC={auc_str}"
            )

        fold_reports.append(fold_metrics)

    final_result = {
        **clean_best_params,
        "mean_score": best_score,
        "std_score": grid_search.cv_results_["std_test_score"][grid_search.best_index_],
        "fold_reports": fold_reports,
        "targets_used": final_target_list,
        "n_splits_used": n_splits,
        "n_rows_used": int(len(df_validi))
    }

    return final_result

# Training AMBL


In [4]:
def training_ambl(file_path: Path, csv_name: str):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    # --- Controllo colonne target attese ---
    required_targets = ["ER [SII]", "PR [SII]", "HER2 [SII]"]
    missing = [c for c in required_targets if c not in df_validi.columns]
    if missing:
        print(f"[ERRORE] {csv_name} - mancano colonne target: {missing}")
        return None

    # --- Creo i target binari direttamente (già binari nel Duke) ---
    final_target_list = ["ER_class", "PR_class", "HER2_class"]

    df_validi["ER_class"]   = pd.to_numeric(df_validi["ER [SII]"], errors="coerce")
    df_validi["PR_class"]   = pd.to_numeric(df_validi["PR [SII]"], errors="coerce")
    df_validi["HER2_class"] = pd.to_numeric(df_validi["HER2 [SII]"], errors="coerce")

    # Tengo solo righe con tutti i target presenti e casto a int
    df_validi = df_validi.dropna(subset=final_target_list).copy()
    for col in final_target_list:
        df_validi[col] = df_validi[col].astype(int)

    # --- Tolgo target con 1 sola classe ---
    targets_da_rimuovere = []
    for col in final_target_list:
        if df_validi[col].nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # --- Features / Target / Groups ---
    raw_target_cols = ["ER", "PR", "HER2"]

    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign", "GRADE", "isTN", "Breast"
    ] + raw_target_cols + final_target_list

    features = df_validi.drop(columns=features_to_drop, errors="ignore")
    target = df_validi[final_target_list]
    groups = df_validi["Patient ID"]

    # Imputazione features numeriche
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.fillna(features.mean(numeric_only=True))

    # Pulizia nomi colonne
    features.columns = [re.sub(r"\[|\]|<", "", col) for col in features.columns]

    # --- StratifiedGroupKFold ---
    if "HER2_class" in final_target_list:
        y_strat = target["HER2_class"].astype(str)
        n_pos = int(target["HER2_class"].sum())
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg("_".join, axis=1)
        n_splits = 5


    # Debug
    min_class_count = y_strat.value_counts().min()
    n_splits = min(n_splits, min_class_count)

    if n_splits < 2:
        print(f"[ERRORE] {csv_name} - troppo pochi campioni per CV stratificata")
        return None

    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    if len(rare) > 0:
        y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(sgkf.split(features, y_strat, groups=groups))

    # Debug utilissimo: controlla positivi HER2 per fold
    """if "HER2_class" in final_target_list:
        for k, (tr, te) in enumerate(splits):
            tr_pos = int(target.iloc[tr]["HER2_class"].sum())
            te_pos = int(target.iloc[te]["HER2_class"].sum())
            print(f"[{csv_name}] Fold {k}: HER2 train pos={tr_pos} | test pos={te_pos}")"""

    base_model = HistGradientBoostingClassifier(
        random_state=42,
        class_weight="balanced",
        early_stopping=False,
        l2_regularization=0.0,
        min_samples_leaf=10
    )
    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        "estimator__learning_rate": [0.05, 0.1],
        "estimator__max_iter": [100, 200],
        "estimator__max_depth": [3, 5],
    }

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average="macro", zero_division=0))
        return float(np.mean(scores))

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = math.prod(len(v) for v in iperparametri.values())
    #print(f"\nInizio Grid Search HistGradientBoosting (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=splits,
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=True,
        return_train_score=True,
        error_score="raise"
    )




    grid_search.fit(features, target)


    # Stampo return_train_score
    #print(pd.DataFrame(grid_search.cv_results_))


    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    clean_best_params = {k.replace("estimator__", ""): v for k, v in best_params.items()}

    final_params = {
        "random_state": 42,
        "class_weight": "balanced",
        "early_stopping": False,
        "l2_regularization": 0.0,
        "min_samples_leaf": 10,
        **clean_best_params
    }

    # Metriche per fold
    fold_reports = []
    for k, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = MultiOutputClassifier(HistGradientBoostingClassifier(**final_params))
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)


        # TODO: capire come è fatto. -> ricavare le prob singole per ciascuna etichetta e calcolare la AUC.
        y_proba_list = model_clone.predict_proba(X_test)

        # DEBUG: Fold k
        print(f"\n[DEBUG] Fold {k}")
        for i, col in enumerate(final_target_list):
            proba_i = y_proba_list[i]
            y_true_i = y_test.iloc[:, i].values

            print(
                f"  Target: {col} | "
                f"y_true classes: {np.unique(y_true_i)} | "
                f"proba shape: {proba_i.shape}"
            )




        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i].values
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, average="macro", zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            auc_val = np.nan
            proba_i = y_proba_list[i]

            # Caso BINARIO
            if len(np.unique(y_true_i)) == 2 and proba_i.shape[1] == 2:
                auc_val = roc_auc_score(y_true_i, proba_i[:, 1])

            # Caso MULTICLASSE (AMBL)
            elif len(np.unique(y_true_i)) > 2:
                try:
                    auc_val = roc_auc_score(
                        y_true_i,
                        proba_i,
                        multi_class="ovr",
                        average="macro"
                    )
                except ValueError:
                    auc_val = np.nan

            fold_metrics[col] = {"f1": f1, 
                                 "accuracy": acc, 
                                 "auc": auc_val
                                }
        # Debug
        print(f"\nMetriche Fold {k}")
        for col, m in fold_metrics.items():
            auc_str = "nan" if np.isnan(m["auc"]) else f"{m['auc']:.3f}"
            print(
                f"  {col}: "
                f"F1={m['f1']:.3f} | "
                f"ACC={m['accuracy']:.3f} | "
                f"AUC={auc_str}"
            )
            
        fold_reports.append(fold_metrics)

    final_result = {
        **clean_best_params,
        "mean_score": best_score,
        "std_score": grid_search.cv_results_["std_test_score"][grid_search.best_index_],
        "fold_reports": fold_reports,
        "targets_used": final_target_list,
        "n_splits_used": n_splits,
        "n_rows_used": int(len(df_validi))
    }

    return final_result

# Vado a stampare gli output in una maniera piú leggibile

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path


def print_grid_search_results(results_per_dataset, save_csv=True, output_path="BoostedDecisionTree.csv"):
    print("\n" + "=" * 80)
    print(" " * 20 + "Metriche (MEDIA ± STD) per target")
    print("=" * 80)

    # Per il csv
    rows = []

    for dataset_name, best_result in results_per_dataset.items():
        if best_result is None:
            continue

        fold_reports = best_result["fold_reports"]
        target_names = best_result.get("targets_used", [])

        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        for target_name in target_names:
            f1_list  = np.array([fold[target_name]["f1"] for fold in fold_reports], dtype=float)
            acc_list = np.array([fold[target_name]["accuracy"] for fold in fold_reports], dtype=float)
            auc_list = np.array([fold[target_name]["auc"] for fold in fold_reports], dtype=float)

            f1_mean,  f1_std  = np.mean(f1_list),  np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            valid_auc = ~np.isnan(auc_list)
            auc_mean = np.mean(auc_list[valid_auc]) if valid_auc.any() else np.nan
            auc_std  = np.std(auc_list[valid_auc])  if valid_auc.any() else np.nan

            # ===== STAMPA =====
            """print(f"\nTarget: {target_name}")
            print(f"  F1-score     = {f1_mean:.3f}  ±  {f1_std:.3f}")
            print(f"  Accuracy     = {acc_mean:.3f}  ±  {acc_std:.3f}")
            print(
                f"  AUC          = {auc_mean:.3f}  ±  {auc_std:.3f}"
                if not np.isnan(auc_mean)
                else f"  AUC          = NaN     ±  NaN"
            )"""

            # ===== CSV =====
            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "F1-score": f"{f1_mean:.3f} ± {f1_std:.3f}",
                "Accuracy": f"{acc_mean:.3f} ± {acc_std:.3f}",
                "AUC": (
                    f"{auc_mean:.3f} ± {auc_std:.3f}"
                    if not np.isnan(auc_mean)
                    else "NaN ± NaN"
                )
            })


    # ===== SALVATAGGIO FILE =====
    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        output_path = Path(output_path)
        df_out.to_csv(output_path, index=False)
        print(f"\n Risultati salvati in: {output_path.resolve()}")

# Lettura dei file

In [6]:
start_time = time.time()

# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():

    name_lower = name.lower()

    if "ambl" in name_lower:
        print(f"\n>>> Training AMBL: {name}")
        results_per_dataset[name] = training_ambl(file_path, name)

    elif "duke" in name_lower:
        print(f"\n>>> Training DUKE: {name}")
        results_per_dataset[name] = training_duke(file_path, name)

    else:
        raise ValueError(f"Dataset non riconosciuto: {name}")
    
# Stampa risultati
print_grid_search_results(results_per_dataset)

end_time = time.time()

# Tempo totale
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



>>> Training DUKE: duke_lesions_radiomic
Fitting 5 folds for each of 8 candidates, totalling 40 fits

[DEBUG] Fold 0
  Target: ER_class | y_true classes: [0 1] | proba shape: (59, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (59, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (59, 2)

Metriche Fold 0
  ER_class: F1=0.603 | ACC=0.508 | AUC=0.490
  PR_class: F1=0.441 | ACC=0.441 | AUC=0.477
  HER2_class: F1=0.222 | ACC=0.644 | AUC=0.438

[DEBUG] Fold 1
  Target: ER_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (58, 2)

Metriche Fold 1
  ER_class: F1=0.667 | ACC=0.603 | AUC=0.595
  PR_class: F1=0.392 | ACC=0.466 | AUC=0.448
  HER2_class: F1=0.308 | ACC=0.690 | AUC=0.467

[DEBUG] Fold 2
  Target: ER_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: 